In [1]:
from pydantic import BaseModel

In [2]:
class Student(BaseModel):

    name: str

In [3]:
data = {'name': 'Nikunj'}
stud1 = Student(**data)

In [4]:
stud1

Student(name='Nikunj')

In [ ]:
# Now, if we enter an 'int' in name, it will give us error, which TypedDict was not

stud2 = Student(**{'name': 26})

ValidationError: 1 validation error for Student
name
  Input should be a valid string [type=string_type, input_value=26, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

# We will now see below things
1) `Default value`
2) `Optional Fields`
3) `Coerce`: It has understanding that while creating a class, if we mention any variable as `int`, but then while creating the object we pass `str`, it will automatically convert that `str` to `int`.
4) `Built-in validations`: We can validate Emails using `EmailStr`

 1) Default value

In [7]:
class Student(BaseModel):

    name: str = 'Nikunj'
    age: int = 24

In [8]:
stud1 = Student(**{})

In [9]:
stud1

Student(name='Nikunj', age=24)

**2) Optional fields**

In [10]:
from pydantic import BaseModel
from typing import Optional

In [11]:
class Student(BaseModel):

    name: str = 'Nikunj'
    age: Optional[int] = None

# If using optional, we will have to add default value as None

In [12]:
stud1 = Student(**{"name": 'Jugal'})

print(stud1)

# See age is None

name='Jugal' age=None


`3) Coerce`: See, we mentioned that age should be integer, but passed string but Pydantic converted it to integer and added it in age

In [13]:
class Student(BaseModel):
    name: str
    age: int

In [14]:
stud1 = Student(**
    {
        'name': 'Nikunj',
        'age': '24'
    }
)

print(stud1)

# See it converted str 34 to int 34
print(type(stud1.age))

name='Nikunj' age=24
<class 'int'>


`4) Built-in validation`: Email validation using `EmailStr` class

In [15]:
from pydantic import BaseModel, EmailStr
from typing import Optional

In [16]:
class Student(BaseModel):
    name: str
    age: Optional[int]
    email: EmailStr
    

In [17]:
stud1 = Student(**
               {
                   'name': 'Nikunj',
                   'age': '24', # Will convert it to int and add it into age variable
                   'email': 'nikunjbos26@gmail.com'
               })

In [18]:
print(stud1)
print()

print(stud1.age)

name='Nikunj' age=24 email='nikunjbos26@gmail.com'

24


In [19]:
# Let's mention weird valie in email

stud1 = Student(**
               {
                   'name': 'Nikunj',
                   'age': '24', # Will convert it to int and add it into age variable
                   'email': 'nikunj'
               })

# See, we're getting validation error. for email

ValidationError: 1 validation error for Student
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='nikunj', input_type=str]

`5) Fields`

In [20]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional

In [21]:
class Student(BaseModel):
    name: str
    age: Optional[int] = None
    email: EmailStr
    cgpa: float = Field(gt = 0, lt = 10, default = 5, description = 'Value indicatiing cgpa of the student')

In [22]:
data = {
    'name': 'Nikunj',
    'email': 'nik@gmail.com',
    'cgpa': 8.13
}

stud1 = Student(**data)

In [23]:
print(stud1)
print()

print(stud1.age)

name='Nikunj' age=None email='nik@gmail.com' cgpa=8.13

None


In [25]:
# Entering cgpa greater than 10 will give error

data = {
    'name': 'Nikunj',
    'email': 'nik@gmail.com',
    'cgpa': 11
}

stud1 = Student(**data)

ValidationError: 1 validation error for Student
cgpa
  Input should be less than 10 [type=less_than, input_value=11, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than

`Note`: We can convert this Pydantic object into python dictionary or json

In [26]:
data = {
    'name': 'Nikunj',
    'email': 'nik@gmail.com',
    'cgpa': 8.13
}

stud1 = Student(**data)

In [27]:
d = dict(stud1)

print(d)
print(type(d))

# dict(stud1). This is deprecated. Use model_dump() fpr dictionary
# and model_dump_json() for json

{'name': 'Nikunj', 'age': None, 'email': 'nik@gmail.com', 'cgpa': 8.13}
<class 'dict'>


In [28]:
# To json

json = stud1.model_dump_json()

print(type(json))

<class 'str'>


In [29]:
json

'{"name":"Nikunj","age":null,"email":"nik@gmail.com","cgpa":8.13}'

# Everything combined using Pydantic

In [ ]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal, Annotated

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [ ]:
class Review(BaseModel):

    key_themes: Annotated[List[str], Field(description = "Mention all the key themes discussed in the revieew, into a list")]
    summary: Annotated[str, Field(description = "Summary of the review")]    
    sentiment: Annotated[Literal['Positive', 'Negative', 'Neutral'], Field(description = 'Return sentiment of the review. Either Positive, Negative or Neutral')]
    pros: Annotated[str, Field(description = 'Mention all the pros', default = None)]
    cons: Annotated[str, Field(description = "Mention all the cons", default = None)]    
    name: Annotated[Optional[str], Field(description = 'Name of the reviewer', default = None)]
    email_id: Annotated[Optional[EmailStr], Field(description = 'Email-id of the reviewer', default = None)]

In [5]:
model = ChatOpenAI(
    model = 'gpt-3.5-turbo',
    max_tokens = 300,
    temperature = 0.3
)

In [6]:
structured_model = model.with_structured_output(Review)


C:\Users\jugal\langchain-env\Lib\site-packages\langchain_openai\chat_models\base.py:1669: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [7]:
rev1 = """The Galaxy S23 is a compact yet powerful flagship smartphone.
It offers a premium design, vibrant AMOLED display, and top-tier performance 
thanks to the Snapdragon 8 Gen 2 chip. The phone handles multitasking, gaming, 
and photography smoothly, while the battery comfortably lasts a full day. 
It’s ideal for users who prefer a sleek, one-hand-friendly device without compromising on features.

Pros:
Premium and compact design
Excellent performance with Snapdragon 8 Gen 2
Stunning AMOLED display with 120Hz refresh rate
Great camera performance (especially night mode & video)
Fast charging and full-day battery life
Smooth gaming with minimal heating

Cons:
No charger included in the box
Slightly smaller battery compared to other flagships
Mono speaker sound could be better (if applicable)
Premium price for a small-sized phone
"""

In [8]:
response = structured_model.invoke(rev1)

In [9]:
print(response)
print()

print(response.summary)
print()

print(response.pros)
print()

print(response.cons)
print()

print(response.sentiment)
print()

print(response.name)
print()

print(response.email_id)


key_themes=['compact design', 'premium design', 'AMOLED display', 'Snapdragon 8 Gen 2 chip', 'performance', 'multitasking', 'gaming', 'photography', 'battery life', 'one-hand-friendly', 'fast charging', 'night mode', 'video', 'gaming experience', 'charger not included', 'battery size', 'speaker sound', 'price'] summary='The Galaxy S23 is a compact yet powerful flagship smartphone with a premium design, vibrant AMOLED display, and top-tier performance. It excels in multitasking, gaming, and photography, with a battery that lasts a full day. Ideal for users who prioritize a sleek, one-hand-friendly device without compromising on features.' sentiment='pos' pros=['Premium and compact design', 'Excellent performance with Snapdragon 8 Gen 2', 'Stunning AMOLED display with 120Hz refresh rate', 'Great camera performance (especially night mode & video)', 'Fast charging and full-day battery life', 'Smooth gaming with minimal heating'] cons=['No charger included in the box', 'Slightly smaller bat

In [10]:
# Adding name and email_id

rev2 = """The Galaxy S23 is a compact yet powerful flagship smartphone.
It offers a premium design, vibrant AMOLED display, and top-tier performance 
thanks to the Snapdragon 8 Gen 2 chip. The phone handles multitasking, gaming, 
and photography smoothly, while the battery comfortably lasts a full day. 
It’s ideal for users who prefer a sleek, one-hand-friendly device without compromising on features.

Pros:
Premium and compact design
Excellent performance with Snapdragon 8 Gen 2
Stunning AMOLED display with 120Hz refresh rate
Great camera performance (especially night mode & video)
Fast charging and full-day battery life
Smooth gaming with minimal heating

Cons:
No charger included in the box
Slightly smaller battery compared to other flagships
Mono speaker sound could be better (if applicable)
Premium price for a small-sized phone

Reviewed by: Nikunj Bosamiya
email: nikunjbos26@gmail.com
"""

In [11]:
response = structured_model.invoke(rev2)

In [12]:
print(response)
print()

print(response.summary)
print()

print(response.pros)
print()

print(response.cons)
print()

print(response.sentiment)
print()

print(response.name)
print()

print(response.email_id)


key_themes=['compact design', 'performance', 'display', 'camera', 'battery life', 'gaming', 'charger', 'speaker', 'price'] summary='The Galaxy S23 is a compact yet powerful flagship smartphone with a premium design, vibrant AMOLED display, and top-tier performance. It excels in multitasking, gaming, and photography, with a full-day battery life. However, it lacks a charger in the box and has a premium price for its size.' sentiment='pos' pros=['Premium and compact design', 'Excellent performance with Snapdragon 8 Gen 2', 'Stunning AMOLED display with 120Hz refresh rate', 'Great camera performance (especially night mode & video)', 'Fast charging and full-day battery life', 'Smooth gaming with minimal heating'] cons=['No charger included in the box', 'Slightly smaller battery compared to other flagships', 'Mono speaker sound could be better (if applicable)', 'Premium price for a small-sized phone'] name='Nikunj Bosamiya' email_id='nikunjbos26@gmail.com'

The Galaxy S23 is a compact yet p

In [48]:
# Converting to json

response_dict = response.model_dump()
response_json = response.model_dump_json(indent = 2)

print(response_dict)
print()
print()

print(response_json)

{'key_themes': ['compact design', 'performance', 'display', 'camera', 'battery life', 'gaming', 'charging', 'price'], 'summary': 'The Galaxy S23 is a compact yet powerful flagship smartphone with a premium design, vibrant AMOLED display, and top-tier performance. It excels in multitasking, gaming, and photography, with a battery that lasts a full day. Ideal for users seeking a sleek, one-hand-friendly device without compromising on features.', 'sentiment': 'pos', 'pros': ['Premium and compact design', 'Excellent performance with Snapdragon 8 Gen 2', 'Stunning AMOLED display with 120Hz refresh rate', 'Great camera performance (especially night mode & video)', 'Fast charging and full-day battery life', 'Smooth gaming with minimal heating'], 'cons': ['No charger included in the box', 'Slightly smaller battery compared to other flagships', 'Mono speaker sound could be better (if applicable)', 'Premium price for a small-sized phone'], 'name': 'Nikunj Bosamiya'}


{
  "key_themes": [
    "co

# There're three ways to create output schema:
1) Using TypedDict
2) Using pydantic
3) Using .json file where we create our schema

* `TypedDict` doesn't validate values whereas `pydantic` does, so we will always use `pydantic`.
* `json` is used when we are using two platforms for like `Python and Java`.
* But we will mostly work with Python, so, `we will always use pydantic`.

# In short, we will learn pydantic only

# Now, one more thing, we have two types of models
1) Which supports structured output, like we use here.
2) Which doesn't support structured output.

# Those models which doesn't support structured outputs, we have to use output parsers by ourself. `We will learn this in our next session`